# Heart Disease Detection — All-In-One Pipeline
## Data Preparation → Model Training → Evaluation → Champion Selection

This notebook combines the full CRISP-DM Stage 2 workflow into a single sequential run:
1. **Data Preparation** (from `data_preparation.ipynb`)
2. **Model Training** — Logistic Regression, KNN, XGBoost (from individual notebooks)
3. **Unified Evaluation & Comparison** (from `model_comparison.py`)
4. **Automatic Champion Selection** with interpretation guide

All code is designed to run top-to-bottom in one go.

## 1. Import All Libraries

In [12]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    f1_score,
    accuracy_score,
    precision_recall_curve,
    auc,
    precision_score,
    recall_score,
    brier_score_loss,
    log_loss,
)
from sklearn.calibration import calibration_curve

from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("All libraries imported successfully!")
print(f"Random State: {RANDOM_STATE}")

All libraries imported successfully!
Random State: 42


## 2. Data Preparation (CRISP-DM Phase 3)

This section reproduces the full data preparation pipeline from `data_preparation.ipynb`:
- DP1: Select target and features
- DP5: Train/test split BEFORE preprocessing (leakage prevention)
- DP2: Clean (impute, deduplicate, remove impossible values)
- DP4: Construct (feature engineering + de-leakage)
- DP3: Structure (encode categoricals)

### 2.1 Load Raw Data

In [13]:
df = pd.read_csv("heart_2022_with_nans.csv")

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Replace empty strings/whitespace with NaN
df = df.replace(r'^\s*$', np.nan, regex=True)
original_shape = df.shape

# Missing values summary
missing_values = df.isnull().sum()
missing_pct = (missing_values / len(df)) * 100
missing_summary = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing %": missing_pct.round(2)
})
missing_summary = missing_summary[missing_summary["Missing Count"] > 0].sort_values("Missing %", ascending=False)

if len(missing_summary) > 0:
    print(f"\nColumns with missing values ({len(missing_summary)}):")
    print(missing_summary.to_string())
else:
    print("\nNo missing values found.")

DATASET OVERVIEW
Dataset Shape: 445,132 rows x 40 columns

Columns with missing values (38):
                           Missing Count  Missing %
TetanusLast10Tdap                  82516      18.54
PneumoVaxEver                      77040      17.31
HIVTesting                         66127      14.86
ChestScan                          56046      12.59
CovidPos                           50764      11.40
HighRiskLastYear                   50623      11.37
BMI                                48806      10.96
FluVaxLast12                       47121      10.59
AlcoholDrinkers                    46574      10.46
WeightInKilograms                  42078       9.45
ECigaretteUsage                    35660       8.01
SmokerStatus                       35462       7.97
HeightInMeters                     28652       6.44
DifficultyErrands                  25656       5.76
DifficultyConcentrating            24240       5.45
DifficultyWalking                  24012       5.39
DifficultyDressingBathi

### 2.2 DP5 — Train/Test Split FIRST (Leakage Prevention)

In [14]:
target_col = "HadHeartAttack"
TEST_SIZE = 0.2

X_raw = df.drop(columns=[target_col])
y_raw = df[target_col]

# Drop rows where target is missing
valid_mask = y_raw.notna()
X_raw = X_raw[valid_mask]
y_raw = y_raw[valid_mask]
print(f"Dropped {(~valid_mask).sum():,} rows with missing target")

# Stratified split on RAW data
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_raw
)

print(f"Training set (raw): {X_train_raw.shape}")
print(f"Test set (raw):     {X_test_raw.shape}")

Dropped 3,065 rows with missing target
Training set (raw): (353653, 39)
Test set (raw):     (88414, 39)


### 2.3 DP2 — Clean: Impute Missing Values (Train-Only Statistics)

In [15]:
X_train_clean = X_train_raw.copy()
X_test_clean = X_test_raw.copy()

numerical_cols = X_train_clean.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X_train_clean.select_dtypes(include=["object", "string"]).columns.tolist()

print("Imputing missing values (statistics from TRAINING set only)...")

# Numerical: median
for col in numerical_cols:
    if X_train_raw[col].isnull().sum() > 0 or X_test_raw[col].isnull().sum() > 0:
        median_val = X_train_clean[col].median()
        train_miss = X_train_clean[col].isnull().sum()
        test_miss = X_test_clean[col].isnull().sum()
        X_train_clean[col] = X_train_clean[col].fillna(median_val)
        X_test_clean[col] = X_test_clean[col].fillna(median_val)
        print(f"  {col}: median={median_val:.2f} (train: {train_miss:,}, test: {test_miss:,})")

# Categorical: mode
for col in categorical_cols:
    if X_train_raw[col].isnull().sum() > 0 or X_test_raw[col].isnull().sum() > 0:
        mode_val = X_train_clean[col].mode()[0]
        train_miss = X_train_clean[col].isnull().sum()
        test_miss = X_test_clean[col].isnull().sum()
        X_train_clean[col] = X_train_clean[col].fillna(mode_val)
        X_test_clean[col] = X_test_clean[col].fillna(mode_val)
        print(f"  {col}: mode='{mode_val}' (train: {train_miss:,}, test: {test_miss:,})")

Imputing missing values (statistics from TRAINING set only)...
  PhysicalHealthDays: median=0.00 (train: 8,456, test: 2,141)
  MentalHealthDays: median=0.00 (train: 7,012, test: 1,780)
  SleepHours: median=7.00 (train: 4,120, test: 1,076)
  HeightInMeters: median=1.70 (train: 22,478, test: 5,663)
  WeightInKilograms: median=80.74 (train: 33,154, test: 8,312)
  BMI: median=27.44 (train: 38,430, test: 9,680)
  GeneralHealth: mode='Very good' (train: 888, test: 207)
  LastCheckupTime: mode='Within past year (anytime less than 12 months ago)' (train: 6,426, test: 1,615)
  PhysicalActivities: mode='Yes' (train: 767, test: 205)
  RemovedTeeth: mode='None of them' (train: 8,820, test: 2,190)
  HadAngina: mode='No' (train: 2,845, test: 743)
  HadStroke: mode='No' (train: 840, test: 230)
  HadAsthma: mode='No' (train: 1,125, test: 312)
  HadSkinCancer: mode='No' (train: 2,237, test: 527)
  HadCOPD: mode='No' (train: 1,448, test: 390)
  HadDepressiveDisorder: mode='No' (train: 1,915, test: 506)


### 2.4 DP2 — Clean: Remove Duplicates & Impossible Values

In [16]:
# Remove duplicates from training set only
dup_count = X_train_clean.duplicated().sum()
if dup_count > 0:
    dup_indices = X_train_clean[X_train_clean.duplicated()].index
    X_train_clean = X_train_clean.drop_duplicates()
    y_train = y_train.drop(dup_indices)
    print(f"Removed {dup_count:,} duplicate rows from training set")
else:
    print("No duplicates found in training set")

# Remove impossible values from training; clip in test
exclusion_rules = {
    "SleepHours": (0, 24),
    "HeightInMeters": (1.0, 2.5),
    "BMI": (10, 60),
}

train_before = len(X_train_clean)
for col, (low, high) in exclusion_rules.items():
    if col in X_train_clean.columns:
        mask = (X_train_clean[col] < low) | (X_train_clean[col] > high)
        n_bad = mask.sum()
        if n_bad > 0:
            X_train_clean = X_train_clean[~mask]
            y_train = y_train.loc[X_train_clean.index]
            print(f"  Train: {col} outside [{low}, {high}]: {n_bad:,} rows removed")

        test_bad = ((X_test_clean[col] < low) | (X_test_clean[col] > high)).sum()
        if test_bad > 0:
            X_test_clean[col] = X_test_clean[col].clip(low, high)
            print(f"  Test:  {col}: {test_bad:,} values clipped to [{low}, {high}]")

print(f"\nTraining set: {train_before:,} -> {len(X_train_clean):,} rows")

Removed 222 duplicate rows from training set
  Train: HeightInMeters outside [1.0, 2.5]: 28 rows removed
  Test:  HeightInMeters: 3 values clipped to [1.0, 2.5]
  Train: BMI outside [10, 60]: 576 rows removed
  Test:  BMI: 144 values clipped to [10, 60]

Training set: 353,431 -> 352,827 rows


### 2.5 DP4 — Feature Construction & De-Leakage

In [17]:
# Feature construction
if "PhysicalHealthDays" in X_train_clean.columns and "MentalHealthDays" in X_train_clean.columns:
    X_train_clean["PoorHealthDaysTotal"] = X_train_clean["PhysicalHealthDays"] + X_train_clean["MentalHealthDays"]
    X_test_clean["PoorHealthDaysTotal"] = X_test_clean["PhysicalHealthDays"] + X_test_clean["MentalHealthDays"]
    print("Created: PoorHealthDaysTotal")

# Drop leakage / redundant / high-cardinality
drop_cols = []
if "HadAngina" in X_train_clean.columns:
    drop_cols.append("HadAngina")
if "State" in X_train_clean.columns:
    drop_cols.append("State")
if "BMI" in X_train_clean.columns:
    for col in ["HeightInMeters", "WeightInKilograms"]:
        if col in X_train_clean.columns:
            drop_cols.append(col)

if drop_cols:
    X_train_clean = X_train_clean.drop(columns=drop_cols)
    X_test_clean = X_test_clean.drop(columns=drop_cols)
    print(f"Dropped columns: {drop_cols}")

print(f"Shapes after DP4 — Train: {X_train_clean.shape}, Test: {X_test_clean.shape}")

Created: PoorHealthDaysTotal
Dropped columns: ['HadAngina', 'State', 'HeightInMeters', 'WeightInKilograms']
Shapes after DP4 — Train: (352827, 36), Test: (88414, 36)


### 2.6 DP3 — Encode Categorical Variables

In [18]:
categorical_cols = X_train_clean.select_dtypes(include=["object", "string"]).columns.tolist()

# Binary columns: label encoding
binary_cols = [c for c in categorical_cols if X_train_clean[c].nunique() == 2]
multi_cols = [c for c in categorical_cols if X_train_clean[c].nunique() > 2]

for col in binary_cols:
    mapping = {cls: idx for idx, cls in enumerate(sorted(X_train_clean[col].dropna().unique()))}
    X_train_clean[col] = X_train_clean[col].map(mapping).fillna(-1).astype(int)
    X_test_clean[col] = X_test_clean[col].map(mapping).fillna(-1).astype(int)

# Multi-category columns: one-hot encoding
for col in multi_cols:
    train_dummies = pd.get_dummies(X_train_clean[col], prefix=col)
    test_dummies = pd.get_dummies(X_test_clean[col], prefix=col)
    for c in train_dummies.columns:
        if c not in test_dummies.columns:
            test_dummies[c] = 0
    test_dummies = test_dummies[train_dummies.columns]

    X_train_clean = X_train_clean.drop(columns=[col])
    X_test_clean = X_test_clean.drop(columns=[col])
    X_train_clean = pd.concat([X_train_clean, train_dummies], axis=1)
    X_test_clean = pd.concat([X_test_clean, test_dummies], axis=1)

print(f"Encoded {len(binary_cols)} binary + {len(multi_cols)} multi-category columns")
print(f"Final shapes — Train: {X_train_clean.shape}, Test: {X_test_clean.shape}")

Encoded 21 binary + 10 multi-category columns
Final shapes — Train: (352827, 76), Test: (88414, 76)


### 2.7 Finalize & Validate

In [19]:
X_train = X_train_clean
X_test = X_test_clean

# Encode target variable
le = LabelEncoder()
le.fit(["No", "Yes"])
y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

# Validation checks
overlap = set(X_train.index).intersection(set(X_test.index))
assert len(overlap) == 0, "Data leakage detected!"
assert X_train.isnull().sum().sum() == 0, "Missing values in training set!"
assert X_test.isnull().sum().sum() == 0, "Missing values in test set!"
assert list(X_train.columns) == list(X_test.columns), "Feature mismatch!"

neg = np.sum(y_train_enc == 0)
pos = np.sum(y_train_enc == 1)
imbalance_ratio = neg / pos

print("=" * 60)
print("DATA PREPARATION COMPLETE")
print("=" * 60)
print(f"Training: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
print(f"Test:     {X_test.shape[0]:,} samples")
print(f"Class ratio (No:Yes): {imbalance_ratio:.2f}:1")
print(f"Positive rate (train): {np.mean(y_train_enc):.4f}")
print("All leakage prevention checks passed.")

DATA PREPARATION COMPLETE
Training: 352,827 samples, 76 features
Test:     88,414 samples
Class ratio (No:Yes): 16.60:1
Positive rate (train): 0.0568
All leakage prevention checks passed.


---
### 2.8 Comparison of Metrics between With & Without Pipeline Architecture

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import cross_val_score

# ── Small reproducible subset ─────────────────────────────────────────────────
DEMO_N = 50000
rng = np.random.default_rng(RANDOM_STATE)
demo_idx = rng.choice(len(X_train), size=min(DEMO_N, len(X_train)), replace=False)
X_demo = X_train.iloc[demo_idx].reset_index(drop=True)
y_demo = y_train_enc[demo_idx]

demo_test_idx = rng.choice(len(X_test), size=min(300, len(X_test)), replace=False)
X_demo_test = X_test.iloc[demo_test_idx].reset_index(drop=True)
y_demo_test = y_test_enc[demo_test_idx]

cv_demo = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ── Custom transformer (fit only on training fold each time) ──────────────────
class TargetMeanFeature(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X_ = X if isinstance(X, np.ndarray) else X.values
        self.col_means_ = X_[:, :5].mean(axis=0)
        return self
    def transform(self, X, y=None):
        X_ = X if isinstance(X, np.ndarray) else X.values
        leak_feature = X_[:, :5].dot(self.col_means_).reshape(-1, 1)
        return np.hstack([X_, leak_feature])

# ── Shared leaky preprocessing (done once, reused across all models) ──────────
y_demo_series = pd.Series(y_demo, name="HadAngina")
target_mean_all = y_demo_series.mean()

X_demo_leaked = X_demo.copy()
X_demo_leaked["HadAngina"] = y_demo_series.values
X_demo_leaked["target_mean_encode"] = X_demo_leaked["HadAngina"].map(
    lambda v: y_demo_series[y_demo_series == v].mean()
)
X_demo_leaked = X_demo_leaked.drop(columns=["HadAngina"])

scaler_leaky = StandardScaler()
X_demo_leaked_scaled = scaler_leaky.fit_transform(X_demo_leaked)

X_demo_test_leaked = X_demo_test.copy()
X_demo_test_leaked["target_mean_encode"] = target_mean_all
X_demo_test_leaked_scaled = scaler_leaky.transform(X_demo_test_leaked)

# ── Imbalance ratio for XGBoost ───────────────────────────────────────────────
demo_neg = (y_demo == 0).sum()
demo_pos = (y_demo == 1).sum()
demo_scale_pos_weight = demo_neg / demo_pos

# ── Model definitions — mirrors 3.1 / 3.2 / 3.3 pipeline patterns ─────────────
#
#   3.1 Logistic Regression: Pipeline([scaler, logreg]) + RandomizedSearchCV
#   3.2 KNN:                 Pipeline([scaler, knn])    + subsample + retrain
#   3.3 XGBoost:             Pipeline([scaler, xgb])    + scale_pos_weight
#
# Each model below pairs a leaky estimator (plain, no Pipeline) with its
# correct Pipeline equivalent so the gap in CV vs holdout scores is visible.

models = {
    # 3.1 pattern — LogisticRegression with class_weight
    "Logistic Regression": {
        "leaky": LogisticRegression(
            max_iter=5000, random_state=RANDOM_STATE, class_weight="balanced"
        ),
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("logreg", LogisticRegression(
                max_iter=5000, random_state=RANDOM_STATE, class_weight="balanced"
            )),
        ]),
    },
    # 3.2 pattern — KNeighborsClassifier (lazy learner, no class_weight)
    "KNN": {
        "leaky": KNeighborsClassifier(n_neighbors=7, weights="distance", n_jobs=-1),
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("knn", KNeighborsClassifier(n_neighbors=7, weights="distance", n_jobs=-1)),
        ]),
    },
    # 3.3 pattern — XGBClassifier with scale_pos_weight for imbalance
    "XGBoost": {
        "leaky": XGBClassifier(
            scale_pos_weight=demo_scale_pos_weight,
            use_label_encoder=False, eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1,
        ),
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("xgb", XGBClassifier(
                scale_pos_weight=demo_scale_pos_weight,
                use_label_encoder=False, eval_metric="logloss",
                random_state=RANDOM_STATE, n_jobs=-1,
            )),
        ]),
    },
}

# ── Evaluate each model (leaky vs pipeline) ───────────────────────────────────
all_results = []

for model_name, model_dict in models.items():

    # --- WITHOUT Pipeline (leaky) ---
    leaky_est = model_dict["leaky"]
    cv_f1_leaky  = cross_val_score(leaky_est, X_demo_leaked_scaled, y_demo, cv=cv_demo, scoring="f1").mean()
    cv_roc_leaky = cross_val_score(leaky_est, X_demo_leaked_scaled, y_demo, cv=cv_demo, scoring="roc_auc").mean()
    cv_acc_leaky = cross_val_score(leaky_est, X_demo_leaked_scaled, y_demo, cv=cv_demo, scoring="accuracy").mean()

    leaky_est.fit(X_demo_leaked_scaled, y_demo)
    y_pred_leaky  = leaky_est.predict(X_demo_test_leaked_scaled)
    y_proba_leaky = leaky_est.predict_proba(X_demo_test_leaked_scaled)[:, 1]

    holdout_f1_leaky  = f1_score(y_demo_test, y_pred_leaky, zero_division=0)
    holdout_roc_leaky = roc_auc_score(y_demo_test, y_proba_leaky)
    holdout_acc_leaky = accuracy_score(y_demo_test, y_pred_leaky)

    # --- WITH Pipeline (correct) ---
    pipe_est = model_dict["pipeline"]
    cv_f1_pipe  = cross_val_score(pipe_est, X_demo, y_demo, cv=cv_demo, scoring="f1").mean()
    cv_roc_pipe = cross_val_score(pipe_est, X_demo, y_demo, cv=cv_demo, scoring="roc_auc").mean()
    cv_acc_pipe = cross_val_score(pipe_est, X_demo, y_demo, cv=cv_demo, scoring="accuracy").mean()

    pipe_est.fit(X_demo, y_demo)
    y_pred_pipe  = pipe_est.predict(X_demo_test)
    y_proba_pipe = pipe_est.predict_proba(X_demo_test)[:, 1]

    holdout_f1_pipe  = f1_score(y_demo_test, y_pred_pipe, zero_division=0)
    holdout_roc_pipe = roc_auc_score(y_demo_test, y_proba_pipe)
    holdout_acc_pipe = accuracy_score(y_demo_test, y_pred_pipe)

    all_results.append({
        "model_name": model_name,
        "cv_acc_leaky": cv_acc_leaky, "cv_f1_leaky": cv_f1_leaky, "cv_roc_leaky": cv_roc_leaky,
        "holdout_acc_leaky": holdout_acc_leaky, "holdout_f1_leaky": holdout_f1_leaky,
        "holdout_roc_leaky": holdout_roc_leaky,
        "cv_acc_pipe": cv_acc_pipe, "cv_f1_pipe": cv_f1_pipe, "cv_roc_pipe": cv_roc_pipe,
        "holdout_acc_pipe": holdout_acc_pipe, "holdout_f1_pipe": holdout_f1_pipe,
        "holdout_roc_pipe": holdout_roc_pipe,
    })

# ── Print results per model ───────────────────────────────────────────────────
# NOTE: holdout used here solely for leakage illustration,
#       not for model selection or tuning.
for r in all_results:
    df = pd.DataFrame({
        "Metric": [
            "CV Accuracy", "CV F1", "CV ROC-AUC",
            "Holdout Accuracy", "Holdout F1", "Holdout ROC-AUC",
        ],
        "Without Pipeline (Leaky)": [
            f"{r['cv_acc_leaky']:.4f}", f"{r['cv_f1_leaky']:.4f}", f"{r['cv_roc_leaky']:.4f}",
            f"{r['holdout_acc_leaky']:.4f}", f"{r['holdout_f1_leaky']:.4f}", f"{r['holdout_roc_leaky']:.4f}",
        ],
        "With Pipeline (Correct)": [
            f"{r['cv_acc_pipe']:.4f}", f"{r['cv_f1_pipe']:.4f}", f"{r['cv_roc_pipe']:.4f}",
            f"{r['holdout_acc_pipe']:.4f}", f"{r['holdout_f1_pipe']:.4f}", f"{r['holdout_roc_pipe']:.4f}",
        ],
    })
    print("=" * 68)
    print(f"  {r['model_name'].upper()} — LEAKAGE DEMO")
    print(f"  Subset: {len(X_demo):,} train / {len(X_demo_test):,} test")
    print("=" * 68)
    print(df.to_string(index=False))
    print()


---
## 3. Model Training (CRISP-DM Phase 4)

Three candidate models, each inside a `Pipeline(StandardScaler + Estimator)`:
1. **Logistic Regression** — Linear model
2. **KNN** — Distance-based model
3. **XGBoost** — Tree-based ensemble

Each model undergoes:
- RandomizedSearchCV (50 iterations, 5-fold stratified CV, optimising F1)
- Threshold optimisation (sweep 0.10–0.90, select max F1)

### 3.0 Shared Helper: Threshold Optimisation

In [20]:
def find_optimal_threshold(y_true, y_proba, lo=0.10, hi=0.90, step=0.05):
    """Sweep thresholds and return the one that maximises F1-score."""
    best_t, best_f1 = 0.5, 0.0
    results = []
    for t in np.arange(lo, hi + step / 2, step):
        y_pred = (y_proba >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        acc = accuracy_score(y_true, y_pred)
        results.append({"Threshold": round(t, 2), "F1": f1, "Precision": prec, "Recall": rec, "Accuracy": acc})
        if f1 > best_f1:
            best_f1, best_t = f1, round(t, 2)
    return best_t, pd.DataFrame(results)

print("Helper function defined.")

Helper function defined.


### 3.1 Logistic Regression

In [21]:
print("=" * 60)
print("TRAINING: Logistic Regression")
print("=" * 60)

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])

lr_param_dist = {
    "logreg__C": np.logspace(-3, 2, 20),
    "logreg__class_weight": [
        "balanced", {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4},
        {0: 1, 1: 5}, {0: 1, 1: 6}, {0: 1, 1: 8}, {0: 1, 1: 10},
    ],
    "logreg__solver": ["liblinear", "saga"],
    "logreg__penalty": ["l1", "l2"],
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

lr_search = RandomizedSearchCV(
    estimator=lr_pipeline,
    param_distributions=lr_param_dist,
    n_iter=50, scoring="f1", cv=cv_strategy,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1, return_train_score=True,
)
lr_search.fit(X_train, y_train_enc)

lr_best = lr_search.best_estimator_
lr_cv_f1_mean = lr_search.cv_results_["mean_test_score"][lr_search.best_index_]
lr_cv_f1_std = lr_search.cv_results_["std_test_score"][lr_search.best_index_]

print(f"\nBest CV F1: {lr_cv_f1_mean:.4f} +/- {lr_cv_f1_std:.4f}")
print(f"Best params: {lr_search.best_params_}")

# Threshold optimisation
lr_proba = lr_best.predict_proba(X_test)[:, 1]
lr_threshold, lr_thresh_df = find_optimal_threshold(y_test_enc, lr_proba)
print(f"Optimal threshold: {lr_threshold}")

TRAINING: Logistic Regression
Fitting 5 folds for each of 50 candidates, totalling 250 fits

Best CV F1: 0.3250 +/- 0.0035
Best params: {'logreg__solver': 'liblinear', 'logreg__penalty': 'l1', 'logreg__class_weight': {0: 1, 1: 5}, 'logreg__C': np.float64(0.02069138081114789)}
Optimal threshold: 0.5


### 3.2 K-Nearest Neighbors

In [22]:
print("=" * 60)
print("TRAINING: KNN")
print("=" * 60)

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_jobs=-1)),
])

knn_param_dist = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 15, 21],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan", "minkowski"],
}

# Subsample for KNN tuning (lazy learner — full data is too slow for CV)
TUNE_SAMPLE_SIZE = 30000
tune_idx = np.random.choice(len(X_train), size=TUNE_SAMPLE_SIZE, replace=False)
X_tune = X_train.iloc[tune_idx]
y_tune = y_train_enc[tune_idx]

print(f"Using {TUNE_SAMPLE_SIZE:,} samples for hyperparameter tuning...")

knn_search = RandomizedSearchCV(
    estimator=knn_pipeline,
    param_distributions=knn_param_dist,
    n_iter=50, scoring="f1", cv=cv_strategy,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1, return_train_score=True,
)
knn_search.fit(X_tune, y_tune)

knn_cv_f1_mean = knn_search.cv_results_["mean_test_score"][knn_search.best_index_]
knn_cv_f1_std = knn_search.cv_results_["std_test_score"][knn_search.best_index_]

# Retrain on full data with best params
best_knn_params = {k.replace("knn__", ""): v for k, v in knn_search.best_params_.items()}
knn_best = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(**best_knn_params, n_jobs=-1)),
])
knn_best.fit(X_train, y_train_enc)

print(f"\nBest CV F1 (subsample): {knn_cv_f1_mean:.4f} +/- {knn_cv_f1_std:.4f}")
print(f"Best params: {knn_search.best_params_}")
print(f"Retrained on full {len(X_train):,} samples.")

# Threshold optimisation
knn_proba = knn_best.predict_proba(X_test)[:, 1]
knn_threshold, knn_thresh_df = find_optimal_threshold(y_test_enc, knn_proba)
print(f"Optimal threshold: {knn_threshold}")

TRAINING: KNN
Using 30,000 samples for hyperparameter tuning...
Fitting 5 folds for each of 42 candidates, totalling 210 fits

Best CV F1 (subsample): 0.0981 +/- 0.0170
Best params: {'knn__weights': 'uniform', 'knn__n_neighbors': 3, 'knn__metric': 'manhattan'}
Retrained on full 352,827 samples.
Optimal threshold: 0.1


### 3.3 XGBoost

In [23]:
print("=" * 60)
print("TRAINING: XGBoost")
print("=" * 60)

scale_pos_weight = neg / pos  # from data prep section

xgb_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("xgb", XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

xgb_param_dist = {
    "xgb__n_estimators": [100, 200, 300, 400, 500],
    "xgb__max_depth": [3, 4, 5, 6, 7, 8, 10],
    "xgb__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2, 0.3],
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_dist,
    n_iter=50, scoring="f1", cv=cv_strategy,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1, return_train_score=True,
)
xgb_search.fit(X_train, y_train_enc)

xgb_best = xgb_search.best_estimator_
xgb_cv_f1_mean = xgb_search.cv_results_["mean_test_score"][xgb_search.best_index_]
xgb_cv_f1_std = xgb_search.cv_results_["std_test_score"][xgb_search.best_index_]

print(f"\nBest CV F1: {xgb_cv_f1_mean:.4f} +/- {xgb_cv_f1_std:.4f}")
print(f"Best params: {xgb_search.best_params_}")

# Threshold optimisation
xgb_proba = xgb_best.predict_proba(X_test)[:, 1]
xgb_threshold, xgb_thresh_df = find_optimal_threshold(y_test_enc, xgb_proba)
print(f"Optimal threshold: {xgb_threshold}")

TRAINING: XGBoost
Fitting 5 folds for each of 50 candidates, totalling 250 fits

Best CV F1: 0.2735 +/- 0.0025
Best params: {'xgb__n_estimators': 400, 'xgb__max_depth': 10, 'xgb__learning_rate': 0.03}
Optimal threshold: 0.65


### 3.4 Consolidate Trained Models

In [24]:
# Store all models, thresholds, and probabilities in dicts for unified evaluation
models = {
    "Logistic Regression": lr_best,
    "KNN": knn_best,
    "XGBoost": xgb_best,
}

thresholds = {
    "Logistic Regression": lr_threshold,
    "KNN": knn_threshold,
    "XGBoost": xgb_threshold,
}

probas = {
    "Logistic Regression": lr_proba,
    "KNN": knn_proba,
    "XGBoost": xgb_proba,
}

cv_scores = {
    "Logistic Regression": (lr_cv_f1_mean, lr_cv_f1_std),
    "KNN": (knn_cv_f1_mean, knn_cv_f1_std),
    "XGBoost": (xgb_cv_f1_mean, xgb_cv_f1_std),
}

TARGET_NAMES = ["No Heart Disease (0)", "Heart Disease (1)"]

print("All 3 models consolidated. Ready for evaluation.")
for name in models:
    print(f"  {name}: threshold = {thresholds[name]:.2f}")

All 3 models consolidated. Ready for evaluation.
  Logistic Regression: threshold = 0.50
  KNN: threshold = 0.10
  XGBoost: threshold = 0.65


---
## 4. Unified Evaluation & Comparison (CRISP-DM Phase 5)

All models are evaluated on the **same held-out test set** using the same metrics.
This section produces:
- A comparison metrics table (classification + probability calibration)
- Per-model classification reports
- ROC curves, PR curves, calibration curves
- Probability distribution plots
- Confusion matrices

### 4.1 Comparison Metrics Table

In [25]:
def compute_comparison_metrics(models, thresholds, probas, cv_scores, y_test):
    """Build a single DataFrame comparing all models."""
    rows = []
    for name in models:
        y_proba = probas[name]
        t = thresholds[name]
        y_pred = (y_proba >= t).astype(int)

        precs, recs, _ = precision_recall_curve(y_test, y_proba)
        pr_auc_val = auc(recs, precs)

        cv_mean, cv_std = cv_scores[name]

        rows.append({
            "Model": name,
            "CV F1 (mean +/- std)": f"{cv_mean:.4f} +/- {cv_std:.4f}",
            "Threshold": t,
            "Accuracy": accuracy_score(y_test, y_pred),
            "F1 (Yes)": f1_score(y_test, y_pred),
            "F1 (Macro)": f1_score(y_test, y_pred, average="macro"),
            "Precision (Yes)": precision_score(y_test, y_pred, zero_division=0),
            "Recall (Yes)": recall_score(y_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, y_proba),
            "PR-AUC": pr_auc_val,
            "Brier Score": brier_score_loss(y_test, y_proba),
            "Log Loss": log_loss(y_test, y_proba),
        })
    return pd.DataFrame(rows).set_index("Model")

comparison_df = compute_comparison_metrics(models, thresholds, probas, cv_scores, y_test_enc)

print("=" * 90)
print("  MODEL COMPARISON — CLASSIFICATION + PROBABILITY METRICS")
print("=" * 90)
print(comparison_df.to_string(float_format="{:.4f}".format))
comparison_df.to_csv("model_comparison_results.csv")
print("\nSaved to: model_comparison_results.csv")

  MODEL COMPARISON — CLASSIFICATION + PROBABILITY METRICS
                    CV F1 (mean +/- std)  Threshold  Accuracy  F1 (Yes)  F1 (Macro)  Precision (Yes)  Recall (Yes)  ROC-AUC  PR-AUC  Brier Score  Log Loss
Model                                                                                                                                                     
Logistic Regression    0.3250 +/- 0.0035     0.5000    0.9004    0.3166      0.6314           0.2593        0.4062   0.8389  0.2525       0.0763    0.2611
KNN                    0.0981 +/- 0.0170     0.1000    0.8523    0.2163      0.5674           0.1548        0.3588   0.6227  0.1511       0.0617    1.4325
XGBoost                0.2735 +/- 0.0025     0.6500    0.8760    0.2940      0.6130           0.2173        0.4548   0.8213  0.2242       0.1290    0.3900

Saved to: model_comparison_results.csv


### 4.2 Per-Model Classification Reports

In [26]:
for name in models:
    y_proba = probas[name]
    y_pred = (y_proba >= thresholds[name]).astype(int)
    print(f"\n{'=' * 70}")
    print(f"  {name} (threshold = {thresholds[name]:.2f})")
    print(f"{'=' * 70}")
    print(classification_report(y_test_enc, y_pred, target_names=TARGET_NAMES))


  Logistic Regression (threshold = 0.50)
                      precision    recall  f1-score   support

No Heart Disease (0)       0.96      0.93      0.95     83392
   Heart Disease (1)       0.26      0.41      0.32      5022

            accuracy                           0.90     88414
           macro avg       0.61      0.67      0.63     88414
        weighted avg       0.92      0.90      0.91     88414


  KNN (threshold = 0.10)
                      precision    recall  f1-score   support

No Heart Disease (0)       0.96      0.88      0.92     83392
   Heart Disease (1)       0.15      0.36      0.22      5022

            accuracy                           0.85     88414
           macro avg       0.56      0.62      0.57     88414
        weighted avg       0.91      0.85      0.88     88414


  XGBoost (threshold = 0.65)
                      precision    recall  f1-score   support

No Heart Disease (0)       0.96      0.90      0.93     83392
   Heart Disease (1)       

### 4.3 ROC Curve Comparison

In [27]:
COLORS = {"Logistic Regression": "#1f77b4", "KNN": "#ff7f0e", "XGBoost": "#2ca02c"}

fig, ax = plt.subplots(figsize=(8, 7))
for name in models:
    y_proba = probas[name]
    fpr, tpr, _ = roc_curve(y_test_enc, y_proba)
    score = roc_auc_score(y_test_enc, y_proba)
    ax.plot(fpr, tpr, linewidth=2, color=COLORS[name], label=f"{name} (AUC = {score:.4f})")

ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random Classifier")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve Comparison")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("roc_comparison.png", dpi=150, bbox_inches="tight")
plt.close()

### 4.4 Precision-Recall Curve Comparison

In [28]:
fig, ax = plt.subplots(figsize=(8, 7))
prevalence = np.mean(y_test_enc)

for name in models:
    y_proba = probas[name]
    precs, recs, _ = precision_recall_curve(y_test_enc, y_proba)
    pr_auc_val = auc(recs, precs)
    ax.plot(recs, precs, linewidth=2, color=COLORS[name], label=f"{name} (PR-AUC = {pr_auc_val:.4f})")

ax.axhline(y=prevalence, color="grey", linestyle="--", label=f"Baseline (prevalence = {prevalence:.4f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve Comparison")
ax.legend(loc="upper right")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("pr_comparison.png", dpi=150, bbox_inches="tight")
plt.close()

### 4.5 Calibration Curves

In [29]:
fig, ax = plt.subplots(figsize=(8, 7))
for name in models:
    y_proba = probas[name]
    frac_pos, mean_pred = calibration_curve(y_test_enc, y_proba, n_bins=10, strategy="uniform")
    ax.plot(mean_pred, frac_pos, "s-", linewidth=2, color=COLORS[name], label=name)

ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfectly Calibrated")
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives")
ax.set_title("Calibration Curves")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("calibration_comparison.png", dpi=150, bbox_inches="tight")
plt.close()

### 4.6 Probability Distributions

In [30]:
n_models = len(models)
fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5), sharey=True)

for ax, name in zip(axes, models):
    y_proba = probas[name]
    t = thresholds[name]
    ax.hist(y_proba[y_test_enc == 0], bins=50, alpha=0.6, label="Actual: No", color="steelblue")
    ax.hist(y_proba[y_test_enc == 1], bins=50, alpha=0.6, label="Actual: Yes", color="salmon")
    ax.axvline(t, color="red", linestyle="--", linewidth=2, label=f"Threshold = {t:.2f}")
    ax.set_xlabel("Predicted Probability")
    ax.set_title(name)
    ax.legend(fontsize=8)

axes[0].set_ylabel("Count")
fig.suptitle("Predicted Probability Distributions", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("probability_distributions.png", dpi=150, bbox_inches="tight")
plt.close()

### 4.7 Confusion Matrices

In [31]:
fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))

for ax, name in zip(axes, models):
    y_proba = probas[name]
    y_pred = (y_proba >= thresholds[name]).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y_test_enc, y_pred,
        display_labels=TARGET_NAMES,
        cmap="Blues", ax=ax,
    )
    ax.set_title(f"{name}\n(threshold = {thresholds[name]:.2f})")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.close()

---
## 5. Automatic Champion Model Selection

### How to Interpret the Evaluation Results

The champion is selected using a **weighted scoring system** that reflects the priorities of a medical screening task. Here is what each metric tells you and why it matters:

**Classification Metrics (at the optimised threshold):**
- **F1 (Yes):** Harmonic mean of precision and recall for the positive class. This is the primary metric because it balances catching heart disease cases (recall) with not overwhelming the system with false alarms (precision). Higher is better.
- **Recall (Yes):** Of all actual heart disease cases, how many did the model catch? In medical screening, missing a case (false negative) can be life-threatening, so high recall is critical.
- **Precision (Yes):** Of all cases the model flagged as heart disease, how many actually had it? Low precision means many false alarms, which wastes clinical resources and causes unnecessary anxiety.
- **F1 (Macro):** Average F1 across both classes — ensures the model doesn't completely sacrifice performance on the majority class.
- **Accuracy:** Overall correctness. Less informative here due to class imbalance (a model that always predicts "No" would still get ~92% accuracy).

**Probability Quality Metrics (threshold-independent):**
- **ROC-AUC:** Measures how well the model ranks positive cases above negative cases across *all* thresholds. A score of 0.5 = random, 1.0 = perfect ranking. This tells you about the model's fundamental discrimination ability.
- **PR-AUC:** Like ROC-AUC but focuses on performance for the minority (positive) class. More informative than ROC-AUC when classes are heavily imbalanced (as they are here).
- **Brier Score:** Mean squared error of predicted probabilities vs actual outcomes. Lower = better calibrated probabilities. A model with Brier = 0.05 is giving more trustworthy probability estimates than one with Brier = 0.15.
- **Log Loss:** Penalises confident wrong predictions severely. A model that says "99% heart disease" for a healthy patient gets hammered on log loss. Lower is better.

**Calibration Curve (visual):**
- Shows whether "70% probability" actually means ~70% of those patients have heart disease. A well-calibrated model hugs the diagonal. This matters because doctors may use the raw probability score, not just the yes/no label.

### Scoring Weights

The automatic selection uses these weights to rank models:

| Metric | Weight | Rationale |
|--------|--------|-----------|
| F1 (Yes) | 0.25 | Primary balance metric for imbalanced classification |
| ROC-AUC | 0.20 | Threshold-independent discrimination ability |
| PR-AUC | 0.20 | Minority-class ranking quality |
| Recall (Yes) | 0.15 | Penalise models that miss too many positive cases |
| Brier Score | 0.10 | Probability calibration quality |
| Log Loss | 0.10 | Penalise confident wrong predictions |

Lower-is-better metrics (Brier, Log Loss) are inverted before scoring so that higher weighted score = better model.

In [32]:
def select_champion(comparison_df):
    """
    Automatically select the champion model using a weighted multi-metric score.
    
    The weights reflect a medical screening priority:
    - F1 (Yes) and ranking metrics (ROC/PR-AUC) are weighted highest
    - Recall gets a dedicated weight to penalise models that miss positive cases
    - Probability quality (Brier, Log Loss) ensures trustworthy outputs
    """
    
    # Define weights (must sum to 1.0)
    weights = {
        "F1 (Yes)":       0.25,
        "ROC-AUC":        0.20,
        "PR-AUC":         0.20,
        "Recall (Yes)":   0.15,
        "Brier Score":    0.10,  # lower is better -> will be inverted
        "Log Loss":       0.10,  # lower is better -> will be inverted
    }
    
    # Normalise each metric to [0, 1] across the candidate models
    scoring_df = pd.DataFrame(index=comparison_df.index)
    
    for metric, weight in weights.items():
        values = comparison_df[metric].astype(float)
        
        if metric in ("Brier Score", "Log Loss"):
            # Invert: lower raw value = better = higher normalised score
            if values.max() == values.min():
                normalised = pd.Series(1.0, index=values.index)
            else:
                normalised = (values.max() - values) / (values.max() - values.min())
        else:
            # Higher raw value = better
            if values.max() == values.min():
                normalised = pd.Series(1.0, index=values.index)
            else:
                normalised = (values - values.min()) / (values.max() - values.min())
        
        scoring_df[f"{metric} (norm)"] = normalised
        scoring_df[f"{metric} (weighted)"] = normalised * weight
    
    scoring_df["Total Score"] = scoring_df[[c for c in scoring_df.columns if "(weighted)" in c]].sum(axis=1)
    
    champion = scoring_df["Total Score"].idxmax()
    
    return champion, scoring_df


champion_name, scoring_details = select_champion(comparison_df)

print("=" * 90)
print("  CHAMPION MODEL SELECTION — WEIGHTED MULTI-METRIC SCORING")
print("=" * 90)
print()
print("Weighted scores per model:")
print(scoring_details[[c for c in scoring_details.columns if "(weighted)" in c] + ["Total Score"]].to_string(float_format="{:.4f}".format))
print()
print(f">>> CHAMPION MODEL: {champion_name} (Total Score = {scoring_details.loc[champion_name, 'Total Score']:.4f}) <<<")
print()

# Show the champion's key metrics
print("Champion's metrics:")
print(comparison_df.loc[champion_name].to_string())

  CHAMPION MODEL SELECTION — WEIGHTED MULTI-METRIC SCORING

Weighted scores per model:
                     F1 (Yes) (weighted)  ROC-AUC (weighted)  PR-AUC (weighted)  Recall (Yes) (weighted)  Brier Score (weighted)  Log Loss (weighted)  Total Score
Model                                                                                                                                                             
Logistic Regression               0.2500              0.2000             0.2000                   0.0741                  0.0784               0.1000       0.9025
KNN                               0.0000              0.0000             0.0000                   0.0000                  0.1000               0.0000       0.1000
XGBoost                           0.1938              0.1837             0.1443                   0.1500                  0.0000               0.0890       0.7608

>>> CHAMPION MODEL: Logistic Regression (Total Score = 0.9025) <<<

Champion's metrics:
CV F1 (me

### 5.1 Champion Summary & Visualisation

In [33]:
# Bar chart of total weighted scores
fig, ax = plt.subplots(figsize=(8, 5))
scores = scoring_details["Total Score"].sort_values(ascending=True)
colors = ["#2ca02c" if name == champion_name else "#aaaaaa" for name in scores.index]
ax.barh(scores.index, scores.values, color=colors, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Weighted Score")
ax.set_title("Champion Model Selection — Weighted Multi-Metric Score")

for i, (name, val) in enumerate(scores.items()):
    label = f"  {val:.4f}" + (" << CHAMPION" if name == champion_name else "")
    ax.text(val, i, label, va="center", fontweight="bold" if name == champion_name else "normal")

plt.tight_layout()
plt.savefig("champion_selection.png", dpi=150, bbox_inches="tight")
plt.close()

print(f"\nChampion model selected: {champion_name}")
print(f"Threshold: {thresholds[champion_name]:.2f}")


Champion model selected: Logistic Regression
Threshold: 0.50


---
## 6. Save Model Artifacts

Save each model as a joblib dict with keys `pipeline`, `threshold`, `label_encoder`
for downstream use (e.g., ablation experiments on the champion).

In [34]:
artifact_paths = {
    "Logistic Regression": "logistic_regression_baseline.joblib",
    "KNN": "knn_baseline.joblib",
    "XGBoost": "xgboost_baseline.joblib",
}

for name, pipeline in models.items():
    artifact = {
        "pipeline": pipeline,
        "threshold": thresholds[name],
        "label_encoder": le,
    }
    path = artifact_paths[name]
    joblib.dump(artifact, path)
    print(f"Saved: {path}")

# Save test data for future use
X_test.to_csv("X_test.csv", index=False)
y_test_series = pd.Series(y_test)
y_test_series.to_csv("y_test.csv", index=False)
X_train.to_csv("X_train.csv", index=False)
y_train_series = pd.Series(y_train)
y_train_series.to_csv("y_train.csv", index=False)

print("\nAll artifacts saved.")
print(f"\nChampion model ({champion_name}) is ready for Part C ablation experiments.")

Saved: logistic_regression_baseline.joblib
Saved: knn_baseline.joblib
Saved: xgboost_baseline.joblib

All artifacts saved.

Champion model (Logistic Regression) is ready for Part C ablation experiments.
